# EU Research Email Network -- Higher-Order Analysis with SHE

This notebook analyses real email communication data from a European research
institution department (89 members, ~12K emails over 802 days), sourced from the
[SNAP email-Eu-core-temporal dataset](https://snap.stanford.edu/data/email-Eu-core-temporal.html).

We use SHE to:
1. Build a hyperstructure from timestamped email interactions
2. Detect communities and find cross-community bridge structures
3. Track bridge formation and group cohesion over time with decay-weighted windows
4. Plot how bridge scores and cohesion evolve

**Data:** Top 20 most-active members, aggregated into monthly bins (812 interaction records).

> Paranjape, Benson & Leskovec. "Motifs in Temporal Networks." WSDM 2017.

In [ ]:
from pathlib import Path
import logging
logging.getLogger('she.diffusion').setLevel(logging.ERROR)

import networkx as nx
import matplotlib.pyplot as plt

from she import (
    SHEHyperstructure,
    SHEConfig,
    rank_entity_diffusers,
    find_bridge_simplices,
    group_cohesion,
    rank_influencers,
    rolling_windows,
    decay_window,
    ranked_items_to_csv,
    bridges_to_csv,
)

## 1. Load data and assign communities

We load the pre-processed JSONL and use Louvain community detection on the
aggregated email graph to assign each member to a community.

In [ ]:
data_path = Path('..') / 'data' / 'eu_email_dept3.jsonl'

config = SHEConfig(max_dimension=2, spectral_k=5)
hs = SHEHyperstructure.from_jsonl(data_path, name='eu_email', config=config)

# Build aggregate graph for community detection
G = nx.Graph()
for rel_key in hs.relations:
    members = sorted(rel_key, key=str)
    if len(members) == 2:
        w = hs.get_relation_attrs(members).get('weight', 1.0)
        G.add_edge(members[0], members[1], weight=w)

communities = nx.community.louvain_communities(G, weight='weight', seed=42)
community_map = {}
for i, comm in enumerate(communities):
    label = f'dept_{chr(65 + i)}'
    for node in comm:
        community_map[node] = label
        hs.add_entity(node, community=label)

print(hs)
print(f'\nCommunities ({len(communities)}):')
for i, comm in enumerate(communities):
    print(f'  dept_{chr(65+i)}: {sorted(comm)} ({len(comm)} members)')

## 2. Global analysis: diffusers, bridges, graph vs simplex

In [ ]:
print('--- Top entity diffusers ---')
for r in rank_entity_diffusers(hs, top_k=8):
    comm = community_map.get(r.target[0], '?')
    print(f'  {r.target[0]:>5s}  score={r.score:.3f}  community={comm}')

print('\n--- Bridge simplices (cross-community email pairs) ---')
bridges = find_bridge_simplices(hs, top_k=8)
for b in bridges:
    print(f'  {sorted(b.members)}  communities={b.communities_spanned}  '
          f'bridge={b.bridge_score:.1f}')

print('\n--- Graph centrality vs simplex diffusion ---')
comp = rank_influencers(hs, top_k=5)
print('Graph (eigenvector):', [(r.target[0], round(r.score, 3)) for r in comp['graph_centrality']])
print('Simplex diffusion:',  [(r.target[0] if r.dimension == 0 else r.target, round(r.score, 3))
                               for r in comp['simplex_diffusion'][:5]])

## 3. Temporal evolution: bridge scores and cohesion over time

We use quarterly rolling windows (window_size=6 months, step=3 months) to track
how bridge structures and a cross-community group's cohesion evolve.

In [ ]:
snapshots = rolling_windows(hs, window_size=6.0, step=3.0, bounds=(0.0, 27.0))

# Pick a cross-community group: top bridge's members
top_bridge_members = sorted(bridges[0].members, key=str) if bridges else []
print(f'Tracking bridge group: {top_bridge_members}')
print(f'Communities: {[community_map.get(m, "?") for m in top_bridge_members]}\n')

# Collect time series
time_points = []
max_bridge_scores = []
cohesion_scores = []
n_bridges_list = []

for start, end, ws in snapshots:
    mid = (start + end) / 2
    time_points.append(mid)

    # propagate community labels to windowed structure
    for entity in ws.entities:
        if entity in community_map:
            ws.add_entity(entity, community=community_map[entity])

    wb = find_bridge_simplices(ws, top_k=5)
    max_bridge_scores.append(wb[0].bridge_score if wb else 0.0)
    n_bridges_list.append(len(wb))

    if all(m in ws.entities for m in top_bridge_members):
        cs = group_cohesion(ws, top_bridge_members)
        cohesion_scores.append(cs.score)
    else:
        cohesion_scores.append(0.0)

    print(f't=[{start:.0f},{end:.0f})  entities={len(ws.entities):>2d}  '
          f'bridges={len(wb)}  top_bridge={max_bridge_scores[-1]:.1f}  '
          f'cohesion={cohesion_scores[-1]:.4f}')

## 4. Plot: bridge scores and cohesion over time

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Top bridge score over time
ax1.plot(time_points, max_bridge_scores, 'o-', color='#2196F3', linewidth=2, markersize=6)
ax1.fill_between(time_points, max_bridge_scores, alpha=0.15, color='#2196F3')
ax1.set_ylabel('Top bridge score')
ax1.set_title('Cross-community bridge strength over time')
ax1.grid(True, alpha=0.3)

# Cohesion score over time
ax2.plot(time_points, cohesion_scores, 's-', color='#FF5722', linewidth=2, markersize=6)
ax2.fill_between(time_points, cohesion_scores, alpha=0.15, color='#FF5722')
ax2.set_ylabel(f'Cohesion score\n({", ".join(top_bridge_members[:3])}...)')
ax2.set_xlabel('Time (months)')
ax2.set_title(f'Group cohesion: top bridge group over time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eu_email_temporal_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eu_email_temporal_plot.png')

## 5. Decay-weighted view

Compare a hard 6-month window with an exponential decay window (half-life = 3 months)
at the same reference point.

In [ ]:
from she import window

ref_time = 18.0  # month 18

# Hard window: last 6 months
hard = window(hs, start=ref_time - 6, end=ref_time)
for entity in hard.entities:
    if entity in community_map:
        hard.add_entity(entity, community=community_map[entity])

# Decay window: half-life 3 months
decayed = decay_window(hs, reference_time=ref_time, half_life=3.0)
for entity in decayed.entities:
    if entity in community_map:
        decayed.add_entity(entity, community=community_map[entity])

print(f'Reference time: month {ref_time:.0f}\n')
print(f'Hard window [12, 18):  {len(hard.entities)} entities, {len(hard.relations)} relations')
print(f'Decay window (hl=3):   {len(decayed.entities)} entities, {len(decayed.relations)} relations')

hard_bridges = find_bridge_simplices(hard, top_k=3)
decay_bridges = find_bridge_simplices(decayed, top_k=3)

print(f'\nHard window bridges:')
for b in hard_bridges:
    print(f'  {sorted(b.members)}  score={b.bridge_score:.1f}')

print(f'Decay window bridges:')
for b in decay_bridges:
    print(f'  {sorted(b.members)}  score={b.bridge_score:.1f}')

## 6. Export results

Save the bridge analysis to CSV for downstream use.

In [ ]:
csv_text = bridges_to_csv(bridges)
print(csv_text[:500])

diffusers_csv = ranked_items_to_csv(rank_entity_diffusers(hs, top_k=10))
print('\n--- Entity diffusers CSV (first 400 chars) ---')
print(diffusers_csv[:400])

## Takeaway

This real-world email network shows that SHE's higher-order analysis reveals
structures invisible to graph centrality alone:

- **Bridge detection** identifies the specific email pairs that connect
  different research sub-groups -- not just the most-emailed individuals.
- **Temporal windowing** shows how these cross-group bridges strengthen or
  weaken over time as collaboration patterns shift.
- **Decay-weighted windows** give a more realistic view than hard cutoffs,
  letting older interactions fade naturally while still contributing context.
- **Export functions** make it easy to feed SHE results into pandas, spreadsheets,
  or dashboards for further analysis.